In [4]:
# ============================================================
# INTEGRATED DIGITAL DATA CRYPTOSYSTEM WITH PERFORMANCE METRICS
# Pipeline: Split -> GHC -> UHC -> Shift128 -> AES-128
# Metrics: Execution Time, Peak Memory, CPU Usage, Overhead, Throughput, Entropy
# ============================================================

!pip install pycryptodome numpy psutil
import os
import time
import math
import psutil
import hashlib
import tracemalloc
import numpy as np
from google.colab import files
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad

# ============================================================
# 1. HASH & UTIL METRICS FUNCTIONS
# ============================================================

def calculate_md5(filename):
    """Menghitung nilai MD5 hash dari sebuah berkas."""
    hash_md5 = hashlib.md5()
    with open(filename, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def calculate_entropy(data_bytes):
    """Menghitung Shannon Entropy dari sekumpulan bytes (Skala 0 - 8)."""
    if not data_bytes:
        return 0.0
    entropy = 0
    len_data = len(data_bytes)
    counts = np.bincount(np.frombuffer(data_bytes, dtype=np.uint8), minlength=256)
    for count in counts:
        if count > 0:
            p = count / len_data
            entropy -= p * math.log2(p)
    return entropy

def derive_aes_key_iv(user_n):
    """Membuat Key 16-byte dan IV 16-byte deterministik untuk AES berbasis input n."""
    key = hashlib.sha256(f"GHC_UHC_KEY_{user_n}".encode()).digest()[:16]
    iv = hashlib.md5(f"GHC_UHC_IV_{user_n}".encode()).digest()
    return key, iv

# ============================================================
# 2. MATRIX & CIPHER ALGORITHMS
# ============================================================

def complete_graph_matrix(N):
    """Membuat matriks ketetanggaan (Adjacency Matrix) Complete Graph K_N: A = J - I."""
    A = np.ones((N, N), dtype=np.int32) - np.eye(N, dtype=np.int32)
    return A % 256

def determinant_complete_graph(N):
    """Menhitung determinan matriks Complete Graph: det(A(K_N)) = (-1)^(N-1) * (N-1)."""
    return ((-1) ** (N - 1)) * (N - 1)

def generate_unimodular_matrix(N, user_n):
    """
    Membuat Matriks Unimodular UHC (det = 1 mod 256) secara konstan
    menggunakan perkalian matriks segitiga bawah dan atas dengan diagonal 1.
    """
    np.random.seed(user_n)
    L = np.tril(np.random.randint(0, 256, (N, N), dtype=np.int32), -1) + np.eye(N, dtype=np.int32)
    U = np.triu(np.random.randint(0, 256, (N, N), dtype=np.int32), 1) + np.eye(N, dtype=np.int32)
    UHC_matrix = np.dot(L, U) % 256
    return UHC_matrix.astype(np.int32)

def mod_inverse_matrix(A, mod=256):
    """Mencari matriks invers modulo 256 menggunakan Gauss-Jordan Elimination."""
    n = A.shape[0]
    A_aug = A.astype(np.int64)
    I = np.eye(n, dtype=np.int64)
    aug = np.concatenate((A_aug, I), axis=1)

    for col in range(n):
        pivot = -1
        for row in range(col, n):
            if np.gcd(int(aug[row, col]), mod) == 1:
                pivot = row
                break
        if pivot == -1:
            raise ValueError(f"Matriks tidak memiliki invers modulo {mod}")

        if pivot != col:
            aug[[col, pivot]] = aug[[pivot, col]]

        pivot_val = int(aug[col, col])
        inv_pivot = pow(pivot_val, -1, mod)
        aug[col] = (aug[col] * inv_pivot) % mod

        for row in range(n):
            if row != col:
                factor = aug[row, col]
                aug[row] = (aug[row] - factor * aug[col]) % mod

    return aug[:, n:].astype(np.int32)

def shift128_encrypt(data):
    return (data.astype(np.int32) + 128) % 256

def shift128_decrypt(data):
    return (data.astype(np.int32) - 128) % 256

def aes_cbc_encrypt(data_bytes, key, iv):
    cipher = AES.new(key, AES.MODE_CBC, iv)
    return cipher.encrypt(pad(data_bytes, AES.block_size))

def aes_cbc_decrypt(ciphertext_bytes, key, iv):
    cipher = AES.new(key, AES.MODE_CBC, iv)
    return unpad(cipher.decrypt(ciphertext_bytes), AES.block_size)

# ============================================================
# 3. CORE PROCESS: ENCRYPTION & DECRYPTION (WITH METRICS)
# ============================================================

def print_performance_report(title, elapsed_time, peak_memory_kb, cpu_usage, orig_size, proc_size, orig_entropy, proc_entropy):
    throughput_mbps = (orig_size / (1024 * 1024)) / elapsed_time if elapsed_time > 0 else 0.0
    overhead = proc_size - orig_size

    print("\n" + "=" * 80)
    print(f" METRIKS EVALUASI PERFORMA METODE ({title.upper()})")
    print("=" * 80)
    print(f" [1] Waktu Eksekusi         : {elapsed_time:.4f} detik")
    print(f" [2] Throughput             : {throughput_mbps:.2f} MB/detik")
    print(f" [3] Peak Memory (RAM)      : {peak_memory_kb:.2f} KB ({peak_memory_kb / 1024:.2f} MB)")
    print(f" [4] Penggunaan CPU         : {cpu_usage:.1f} %")
    print(f" [5] Ukuran File Awal       : {orig_size} bytes")
    print(f" [6] Ukuran File Hasil      : {proc_size} bytes (Overhead: {overhead:+d} bytes)")
    print(f" [7] Shannon Entropy (Awal) : {orig_entropy:.4f} / 8.0")
    print(f" [8] Shannon Entropy (Hasil): {proc_entropy:.4f} / 8.0")
    print("=" * 80)

def encrypt_file(file_path, user_n):
    print("=" * 80)
    print("PROSES ENKRIPSI TERINTEGRASI (GHC + UHC + Split + Shift128 + AES)")
    print("=" * 80)

    # Inisialisasi Monitoring Performa
    tracemalloc.start()
    process = psutil.Process(os.getpid())
    process.cpu_percent(interval=None) # Flush awal CPU usage
    start_time = time.time()

    N = 2 * user_n
    detK = determinant_complete_graph(N)

    print(f"\n[Info] Input n: {user_n} | Effective N (Block Size): {N}")
    print(f"[Info] Determinan Matriks Graph K_N: {detK} (gcd(det, 256) = {np.gcd(abs(detK), 256)})")

    filename, ext = os.path.splitext(file_path)
    plaintext_raw = open(file_path, "rb").read()
    orig_size = len(plaintext_raw)
    orig_entropy = calculate_entropy(plaintext_raw)

    plaintext_np = np.frombuffer(plaintext_raw, dtype=np.uint8)

    # --- 1. PROSES SPLIT ---
    len_shift = plaintext_np.size % N
    len_hill = plaintext_np.size - len_shift
    print(f"\n[1] Split Data -> Block Hill: {len_hill} bytes | Shift128 Sisa: {len_shift} bytes")

    # --- 2. ENKRIPSI HILL (GHC + UHC) ---
    hill_part = plaintext_np[:len_hill]
    if len_hill > 0:
        hill_blocks = hill_part.reshape((N, int(len_hill / N)))

        K_matrix = complete_graph_matrix(N)
        U_matrix = generate_unimodular_matrix(N, user_n)
        Combined_Key = np.dot(K_matrix, U_matrix) % 256

        hill_cipher = np.dot(Combined_Key, hill_blocks.astype(np.int32)) % 256
        hill_cipher = hill_cipher.reshape(len_hill).astype(np.uint8)
    else:
        hill_cipher = np.array([], dtype=np.uint8)

    # --- 3. ENKRIPSI SHIFT128 ---
    shift_part = plaintext_np[len_hill:]
    shift_cipher = shift128_encrypt(shift_part).astype(np.uint8) if len_shift > 0 else np.array([], dtype=np.uint8)

    # --- 4. COMBINE CIPHER ---
    combined_cipher = np.concatenate((hill_cipher, shift_cipher)).tobytes()

    # --- 5. ENKRIPSI AKHIR: AES-128-CBC ---
    key_aes, iv_aes = derive_aes_key_iv(user_n)
    final_ciphertext = aes_cbc_encrypt(combined_cipher, key_aes, iv_aes)

    output_file = f"{filename}_encrypted{ext}"
    with open(output_file, "wb") as f:
        f.write(final_ciphertext)

    # Hentikan Monitoring & Hitung Metriks
    elapsed_time = time.time() - start_time
    _, peak_mem_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    cpu_usage = process.cpu_percent(interval=None)

    proc_size = len(final_ciphertext)
    proc_entropy = calculate_entropy(final_ciphertext)

    print(f"\n[Status] Enkripsi Selesai")
    print(f"[File] Output Encrypted : {output_file}")
    print(f"[MD5] Original          : {calculate_md5(file_path)}")
    print(f"[MD5] Encrypted         : {calculate_md5(output_file)}")

    # Tampilkan Ringkasan Metriks Evaluasi
    print_performance_report("Enkripsi", elapsed_time, peak_mem_bytes / 1024, cpu_usage, orig_size, proc_size, orig_entropy, proc_entropy)
    return output_file


def decrypt_file(file_path, user_n):
    print("=" * 80)
    print("PROSES DEKRIPSI TERINTEGRASI (AES -> Shift128 + UHC + GHC -> Recombine)")
    print("=" * 80)

    # Inisialisasi Monitoring Performa
    tracemalloc.start()
    process = psutil.Process(os.getpid())
    process.cpu_percent(interval=None)
    start_time = time.time()

    N = 2 * user_n

    filename, ext = os.path.splitext(file_path)
    encrypted_bytes = open(file_path, "rb").read()
    orig_size = len(encrypted_bytes)
    orig_entropy = calculate_entropy(encrypted_bytes)

    # --- 1. DEKRIPSI AES-128-CBC ---
    key_aes, iv_aes = derive_aes_key_iv(user_n)
    decrypted_aes_bytes = aes_cbc_decrypt(encrypted_bytes, key_aes, iv_aes)
    ciphertext_np = np.frombuffer(decrypted_aes_bytes, dtype=np.uint8)

    # --- 2. SPLIT REVERSE ---
    len_shift = ciphertext_np.size % N
    len_hill = ciphertext_np.size - len_shift
    print(f"\n[1] Reverse Split -> Block Hill: {len_hill} bytes | Shift128 Sisa: {len_shift} bytes")

    # --- 3. DEKRIPSI HILL (UHC + GHC INVERSE) ---
    hill_cipher_part = ciphertext_np[:len_hill]
    if len_hill > 0:
        hill_blocks = hill_cipher_part.reshape((N, int(len_hill / N)))

        K_matrix = complete_graph_matrix(N)
        U_matrix = generate_unimodular_matrix(N, user_n)
        Combined_Key = np.dot(K_matrix, U_matrix) % 256

        Inv_Combined_Key = mod_inverse_matrix(Combined_Key)

        hill_plain = np.dot(Inv_Combined_Key, hill_blocks.astype(np.int32)) % 256
        hill_plain = hill_plain.reshape(len_hill).astype(np.uint8)
    else:
        hill_plain = np.array([], dtype=np.uint8)

    # --- 4. DEKRIPSI SHIFT128 ---
    shift_cipher_part = ciphertext_np[len_hill:]
    shift_plain = shift128_decrypt(shift_cipher_part).astype(np.uint8) if len_shift > 0 else np.array([], dtype=np.uint8)

    # --- 5. RECOMBINE PLAINTEXT ---
    plaintext_final = np.concatenate((hill_plain, shift_plain)).tobytes()

    output_file = f"{filename}_decrypted{ext}"
    with open(output_file, "wb") as f:
        f.write(plaintext_final)

    # Hentikan Monitoring & Hitung Metriks
    elapsed_time = time.time() - start_time
    _, peak_mem_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    cpu_usage = process.cpu_percent(interval=None)

    proc_size = len(plaintext_final)
    proc_entropy = calculate_entropy(plaintext_final)

    print(f"\n[Status] Dekripsi Selesai")
    print(f"[File] Output Decrypted : {output_file}")
    print(f"[MD5] Encrypted         : {calculate_md5(file_path)}")
    print(f"[MD5] Decrypted         : {calculate_md5(output_file)}")

    # Tampilkan Ringkasan Metriks Evaluasi
    print_performance_report("Dekripsi", elapsed_time, peak_mem_bytes / 1024, cpu_usage, orig_size, proc_size, orig_entropy, proc_entropy)
    return output_file

# ============================================================
# 4. MAIN INTERACTIVE MENU
# ============================================================

if __name__ == "__main__":
    print("=" * 80)
    print("HYBRID CRYPTOSYSTEM: GHC + UHC + SPLIT + SHIFT128 + AES (WITH METRICS EVALUATION)")
    print("=" * 80)
    print("Pilih Mode:")
    print("e = Encrypt")
    print("d = Decrypt")

    mode = input("\nPilihan Anda (e/d): ").strip().lower()

    if mode == 'e':
        print("\nUpload file digital asli yang akan dienkripsi:")
        uploaded = files.upload()
        file_name = list(uploaded.keys())[0]
        n_val = int(input("Masukkan nilai kunci n (contoh: 2, 4, 8): "))

        encrypted_file = encrypt_file(file_name, n_val)
        print("\nMengunduh berkas terenkripsi...")
        files.download(encrypted_file)

    elif mode == 'd':
        print("\nUpload file terenkripsi yang akan didekripsi:")
        uploaded = files.upload()
        file_name = list(uploaded.keys())[0]
        n_val = int(input("Masukkan nilai kunci n yang SAMA saat enkripsi: "))

        decrypted_file = decrypt_file(file_name, n_val)
        print("\nMengunduh berkas hasil dekripsi...")
        files.download(decrypted_file)

    else:
        print("\nPilihan tidak valid.")

HYBRID CRYPTOSYSTEM: GHC + UHC + SPLIT + SHIFT128 + AES (WITH METRICS EVALUATION)
Pilih Mode:
e = Encrypt
d = Decrypt

Pilihan Anda (e/d): d

Upload file terenkripsi yang akan didekripsi:


Saving FileAttachment (1)_encrypted.pdf to FileAttachment (1)_encrypted (1).pdf
Masukkan nilai kunci n yang SAMA saat enkripsi: 33
PROSES DEKRIPSI TERINTEGRASI (AES -> Shift128 + UHC + GHC -> Recombine)

[1] Reverse Split -> Block Hill: 104214 bytes | Shift128 Sisa: 53 bytes

[Status] Dekripsi Selesai
[File] Output Decrypted : FileAttachment (1)_encrypted (1)_decrypted.pdf
[MD5] Encrypted         : 9fad468d75596eff00c6f9c71ac5939f
[MD5] Decrypted         : 45cba889e00b955da699e7c9361262e3

 METRIKS EVALUASI PERFORMA METODE (DEKRIPSI)
 [1] Waktu Eksekusi         : 0.1824 detik
 [2] Throughput             : 0.55 MB/detik
 [3] Peak Memory (RAM)      : 1095.19 KB (1.07 MB)
 [4] Penggunaan CPU         : 98.5 %
 [5] Ukuran File Awal       : 104272 bytes
 [6] Ukuran File Hasil      : 104267 bytes (Overhead: -5 bytes)
 [7] Shannon Entropy (Awal) : 7.9981 / 8.0
 [8] Shannon Entropy (Hasil): 7.3840 / 8.0

Mengunduh berkas hasil dekripsi...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>